# Embed & Ingest Documentation into Vector Database

This notebook takes the cleaned documentation chunks produced by the **Collect_Documentation_for_VectorDB** notebook and:

1. Loads the chunks from `library_docs_for_vectordb.json`
2. Generates dense embeddings using **BGE-base-en-v1.5** (`BAAI/bge-base-en-v1.5`)
3. Ingests them into a persistent **ChromaDB** vector store with metadata filtering

### Design choices

| Component | Choice | Why |
|-----------|--------|-----|
| **Embedding model** | `BAAI/bge-base-en-v1.5` | Top-tier retrieval performance (MTEB), 768-dim, fast on CPU, works great with technical / code documentation |
| **Vector database** | ChromaDB (persistent) | Lightweight, Python-native, supports metadata filtering (`library`, `version`), easy to integrate with LangChain |
| **LLM (downstream)** | Qwen 2.5 Coder | The retriever feeds context to this model for code correction |

### Metadata filters available at query time
- `library` — retrieve only docs for a specific package (e.g. `"pandas"`)
- `version` — pin to a release (e.g. `"3.0.0"`) or `"latest"`

## 1 — Install & Import Dependencies

In [ ]:
# Install required packages
#!pip install chromadb sentence-transformers tqdm -q

In [3]:
import json
from pathlib import Path
from tqdm.auto import tqdm

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

print(f"ChromaDB version: {chromadb.__version__}")
print("All imports successful.")

c:\Users\hbahmanyar\MentorApp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChromaDB version: 1.5.1
All imports successful.


## 2 — Load Documentation Chunks

In [4]:
DOCS_PATH = Path("../Datasets/library_docs_for_vectordb.json")

with open(DOCS_PATH, encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} document chunks from {DOCS_PATH.name}")
print(f"Libraries : {len(set(d['library'] for d in documents))}")
print(f"Versions  : {len(set(d['version'] for d in documents))} unique")

# Quick stats
total_chars = sum(d["char_count"] for d in documents)
avg_chars = total_chars / len(documents) if documents else 0
print(f"\nTotal characters : {total_chars:,}")
print(f"Avg chunk size   : {avg_chars:,.0f} chars")

Loaded 647 document chunks from library_docs_for_vectordb.json
Libraries : 25
Versions  : 17 unique

Total characters : 804,087
Avg chunk size   : 1,243 chars


## 3 — Initialize Embedding Model

We use **BGE-base-en-v1.5** from Beijing Academy of AI (BAAI) — a top-performing embedding model on the MTEB benchmark.  
It produces 768-dimensional vectors and handles technical documentation / code-related text well.

> **Why BGE over OpenAI embeddings?**  
> Runs locally (no API key needed), reproducible, free, and performs competitively with `text-embedding-3-small` for retrieval tasks.

In [5]:
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# ChromaDB's built-in wrapper handles batching and device selection
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    device="cpu",                  
    normalize_embeddings=True,    
)

# Verify dimensions
test_embedding = embedding_fn(["test"])
EMBED_DIM = len(test_embedding[0])
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Dimensions      : {EMBED_DIM}")
print(f"Device          : cpu")

c:\Users\hbahmanyar\MentorApp\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hbahmanyar\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 720.88it/s, Materializing param=pooler.dense.weigh

Embedding model : BAAI/bge-base-en-v1.5
Dimensions      : 768
Device          : cpu


## 4 — Create ChromaDB Collection

ChromaDB stores:
- **Embeddings** — dense vectors for similarity search
- **Documents** — the raw text chunks
- **Metadata** — `library`, `version`, `source_url`, etc. for filtering

We use **persistent storage** so the database survives kernel restarts and can be loaded by the RAG pipeline notebooks directly.

In [6]:
CHROMA_PERSIST_DIR = str(Path("../VectorDB/chroma_library_docs").resolve())
COLLECTION_NAME = "library_docs"

# Initialize persistent client
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# Create (or get) the collection with the BGE embedding function
# Using cosine distance — standard for normalized BGE embeddings
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},  # distance metric
)

print(f"ChromaDB path   : {CHROMA_PERSIST_DIR}")
print(f"Collection      : {COLLECTION_NAME}")
print(f"Existing docs   : {collection.count()}")

ChromaDB path   : C:\Users\hbahmanyar\MentorApp\VectorDB\chroma_library_docs
Collection      : library_docs
Existing docs   : 0


## 5 — Ingest Documents

ChromaDB computes embeddings automatically via the `embedding_function` we assigned to the collection.  
We add documents in batches to avoid memory spikes and show progress.

Each document is stored with:
- **id** — deterministic hash (prevents duplicates on re-run)
- **document** — the raw text chunk
- **metadata** — `library`, `version`, `source_url`, `chunk_index`, `total_chunks`, `char_count`

In [7]:
BATCH_SIZE = 64  

# Prepare data for ingestion
ids = [doc["id"] for doc in documents]
texts = [doc["text"] for doc in documents]
metadatas = [
    {
        "library": doc["library"],
        "version": doc["version"],
        "source_url": doc["source_url"],
        "chunk_index": doc["chunk_index"],
        "total_chunks": doc["total_chunks"],
        "char_count": doc["char_count"],
    }
    for doc in documents
]

# Ingest in batches with progress bar
for start in tqdm(range(0, len(documents), BATCH_SIZE), desc="Ingesting batches"):
    end = min(start + BATCH_SIZE, len(documents))
    collection.upsert(
        ids=ids[start:end],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
    )

print(f"\n✓ Ingested {len(documents)} documents into '{COLLECTION_NAME}'")
print(f"  Collection now contains: {collection.count()} documents")

Ingesting batches: 100%|██████████| 11/11 [04:48<00:00, 26.19s/it]


✓ Ingested 647 documents into 'library_docs'
  Collection now contains: 647 documents


## 6 — Verify: Basic Retrieval Test

Quick sanity check — query the vector DB with a real code-correction question and see if the right documentation chunks surface.

In [8]:
# ── Test 1: General query (no filter) ────────────────────────────────────────
query = "SettingWithCopyWarning chained assignment pandas"

results = collection.query(
    query_texts=[query],
    n_results=5,
)

print(f"Query: \"{query}\"\n")
print(f"{'─'*70}")
for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
):
    print(f"\n  Result {i+1}  (distance: {dist:.4f})")
    print(f"  Library : {meta['library']} v{meta['version']}")
    print(f"  URL     : {meta['source_url']}")
    print(f"  Preview : {doc[:200]}...")
    print(f"{'─'*70}")

Query: "SettingWithCopyWarning chained assignment pandas"

──────────────────────────────────────────────────────────────────────

  Result 1  (distance: 0.2887)
  Library : pandas v3.0.0
  URL     : https://pandas.pydata.org/docs/whatsnew/v3.0.0.html
  Preview :  this now consistently never works, the SettingWithCopyWarning is removed, and defensive .copy() calls to
silence the warning are no longer needed.
The new behavioral semantics are explained in more d...
──────────────────────────────────────────────────────────────────────

  Result 2  (distance: 0.2973)
  Library : pandas v2.3.0
  URL     : https://pandas.pydata.org/docs/whatsnew/v2.3.0.html
  Preview : nfer_string = True

```

See the Migration guide for the new string data type (pandas 3.0) for more details on the behaviour changes
and how to adapt your code to the new default.

### Copy-on-Write #...
──────────────────────────────────────────────────────────────────────

  Result 3  (distance: 0.3122)
  Library : pandas v

In [9]:
# ── Test 2: Filtered query (by library) ──────────────────────────────────────
query = "deprecated function removed in new version"

results_filtered = collection.query(
    query_texts=[query],
    n_results=5,
    where={"library": "numpy"},                  # filter by library
)

print(f"Query   : \"{query}\"")
print(f"Filter  : library = 'numpy'\n")
print(f"{'─'*70}")
for i, (doc, meta, dist) in enumerate(
    zip(results_filtered["documents"][0],
        results_filtered["metadatas"][0],
        results_filtered["distances"][0])
):
    print(f"\n  Result {i+1}  (distance: {dist:.4f})")
    print(f"  Library : {meta['library']} v{meta['version']}")
    print(f"  Preview : {doc[:200]}...")
    print(f"{'─'*70}")

Query   : "deprecated function removed in new version"
Filter  : library = 'numpy'

──────────────────────────────────────────────────────────────────────

  Result 1  (distance: 0.2946)
  Library : numpy v2.4.0
  Preview : parameter was deprecated in NumPy 1.22.0 and has been
removed from numpy.ma.mrecords.fromtextfile() . Use delimiter instead.
( gh-30021 )

### numpy.array2string and numpy.sum deprecations finalized #...
──────────────────────────────────────────────────────────────────────

  Result 2  (distance: 0.2994)
  Library : numpy v2.4.0
  Preview : np.testing.suppress_warnings are
deprecated. Use warnings.catch_warnings , warnings.filterwarnings , pytest.warns , or pytest.filterwarnings instead.
( gh-29550 )

### np.fix is pending deprecation #
...
──────────────────────────────────────────────────────────────────────

  Result 3  (distance: 0.3009)
  Library : numpy v2.4.0
  Preview : s flag has been ignored since NumPy 1.17 and was only needed to support
loading files in

In [13]:
# ── Test 3: Compound filter (library + version) ─────────────────────────────
query = "how to migrate old code to new API"

results_compound = collection.query(
    query_texts=[query],
    n_results=3,
    where={
        "$and": [
            {"library": "scikit-learn"},
            {"version": "1.8"},
        ]
    },
)

print(f"Query   : \"{query}\"")
print(f"Filter  : library = 'scikit-learn' AND version = '1.6.1'\n")
print(f"{'─'*70}")
for i, (doc, meta, dist) in enumerate(
    zip(results_compound["documents"][0],
        results_compound["metadatas"][0],
        results_compound["distances"][0])
):
    print(f"\n  Result {i+1}  (distance: {dist:.4f})")
    print(f"  Library : {meta['library']} v{meta['version']}")
    print(f"  Preview : {doc[:200]}...")
    print(f"{'─'*70}")

Query   : "how to migrate old code to new API"
Filter  : library = 'scikit-learn' AND version = '1.6.1'

──────────────────────────────────────────────────────────────────────

  Result 1  (distance: 0.4103)
  Library : scikit-learn v1.8
  Preview : # Version 1.8 #

For a short description of the main highlights of the release, please refer to Release Highlights for scikit-learn 1.8 .
Legend for changelogs
Major Feature something big that you cou...
──────────────────────────────────────────────────────────────────────

  Result 2  (distance: 0.4300)
  Library : scikit-learn v1.8
  Preview : eter is deprecated in favour of name in metrics.PrecisionRecallDisplay and will be removed in 1.10.
By Lucy Liu . #32310...
──────────────────────────────────────────────────────────────────────

  Result 3  (distance: 0.4311)
  Library : scikit-learn v1.8
  Preview : n ndarray of shape (n_folds, n_l1_ratios, n_cs). In version 1.10, the default will change to False and use_legacy_attributes will
be

## 7 — Collection Statistics

In [14]:
from collections import Counter

# Fetch all metadata to compute stats
all_meta = collection.get(include=["metadatas"])["metadatas"]

lib_counts = Counter(m["library"] for m in all_meta)
ver_counts = Counter(f"{m['library']} v{m['version']}" for m in all_meta)

print(f"Total documents in collection: {collection.count()}\n")

print("Documents per library:")
for lib, cnt in lib_counts.most_common():
    print(f"  {lib:20s}  {cnt:4d}")

print(f"\nTop 15 library+version combinations:")
for key, cnt in ver_counts.most_common(15):
    print(f"  {key:35s}  {cnt:4d}")

Total documents in collection: 647

Documents per library:
  pandas                 117
  scipy                  116
  numpy                   55
  requests                52
  scikit-learn            51
  matplotlib              44
  openpyxl                32
  torchvision             31
  pillow                  25
  langchain-openai        25
  torch                   21
  datasets                14
  seaborn                 11
  tensorflow              10
  accelerate               9
  httpx                    8
  torchaudio               5
  xgboost                  5
  lightgbm                 5
  scikit-image             4
  langchain                2
  catboost                 2
  transformers             1
  tqdm                     1
  opencv-python            1

Top 15 library+version combinations:
  pandas v3.0.0                         110
  scipy v1.17.0                          61
  scipy v1.16.0                          55
  requests vlatest                       52
  

## 8 — Helper: `retrieve_context()` for RAG Pipeline

A reusable function that the RAG notebooks can import or copy.  
It retrieves relevant documentation context given a code-correction query, with optional metadata filters.

In [15]:
def retrieve_context(
    query: str,
    collection: chromadb.Collection = collection,
    n_results: int = 5,
    library: str | None = None,
    version: str | None = None,
) -> str:
    """
    Retrieve relevant documentation context from the vector DB.

    Parameters
    ----------
    query : str
        The code-correction question or error description.
    n_results : int
        Number of top-k chunks to retrieve.
    library : str, optional
        Filter by package name (e.g. "pandas").
    version : str, optional
        Filter by release version (e.g. "3.0.0" or "latest").

    Returns
    -------
    str
        Concatenated context string ready to inject into the LLM prompt.
    """
    # Build where filter
    filters = []
    if library:
        filters.append({"library": library})
    if version:
        filters.append({"version": version})

    where = None
    if len(filters) == 1:
        where = filters[0]
    elif len(filters) > 1:
        where = {"$and": filters}

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where,
    )

    # Format context for the LLM
    context_parts = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        header = f"[{meta['library']} v{meta['version']}]"
        context_parts.append(f"{header}\n{doc}")

    return "\n\n---\n\n".join(context_parts)


# ── Demo ────────────────────────────────────────────────────────────────────
demo_context = retrieve_context(
    query="numpy.in1d deprecated removed what to use instead",
    library="numpy",
)
print("Retrieved context (first 800 chars):\n")
print(demo_context[:800])
if len(demo_context) > 800:
    print("\n  [... truncated ...]")

Retrieved context (first 800 chars):

[numpy v2.4.0]
s flag has been ignored since NumPy 1.17 and was only needed to support
loading files in Python 2 that were written in Python 3.
( gh-29984 )

### Removal of four undocumented ndarray.ctypes methods #

Four undocumented methods of the ndarray.ctypes object have been removed:
_ctypes.get_data() (use _ctypes.data instead)
_ctypes.get_shape() (use _ctypes.shape instead)
_ctypes.get_strides() (use _ctypes.strides instead)
_ctypes.get_as_parameter() (use _ctypes._as_parameter_ instead)
These methods have been deprecated since NumPy 1.21.
( gh-29986 )

### Removed newshape parameter from numpy.reshape #

The newshape parameter was deprecated in NumPy 2.1.0 and has been
removed from numpy.reshape . Pass it positionally or use shape= on newer NumPy versions.
( gh-29994 )

### Remova

  [... truncated ...]


## 9 — Summary & Next Steps

**What was done in this notebook:**
- Loaded documentation chunks from `library_docs_for_vectordb.json`
- Generated dense embeddings using `BAAI/bge-base-en-v1.5` (768-dim, normalized)
- Ingested all chunks + metadata into a persistent ChromaDB collection at `VectorDB/chroma_library_docs/`
- Verified retrieval with filtered and unfiltered queries

**How to use in the RAG pipeline:**

```python
# In RAG_with_baseline_Qwen.ipynb or RAG_with_SFT_Qwen.ipynb:

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5",
    normalize_embeddings=True,
)
client = chromadb.PersistentClient(path="../VectorDB/chroma_library_docs")
collection = client.get_collection("library_docs", embedding_function=embedding_fn)

# Query with metadata filtering
results = collection.query(
    query_texts=["your code error description"],
    n_results=5,
    where={"library": "pandas"},  # optional filter
)
```

**Metadata filters available:**
| Filter | Values | Use case |
|--------|--------|----------|
| `library` | `"numpy"`, `"pandas"`, ... | Scope to a specific package |
| `version` | `"2.4.0"`, `"3.0.0"`, `"latest"` | Pin to a release |

**Next steps:**
1. Build the RAG prompt template for Qwen 2.5 Coder
2. Implement the full RAG pipeline (retrieve → augment prompt → generate corrected code)
3. Evaluate retrieval quality and tune `n_results` / filters